### CELL 1 : Import

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

### CELL 2: CREATE WATER MARK TABLE

In [0]:
spark.sql('''
          CREATE TABLE IF NOT EXISTS mia_catalog.gold._pipeline_state(
              source_table STRING,
              target_table STRING,
              last_processed_version INT,
              updated_at TIMESTAMP
          )
          USING DELTA
        ''')
print("mia_catalog.gold._pipeline_state ready")

mia_catalog.gold._pipeline_state ready


### DIM_CUSTOMETS TABLE CREATION

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS mia_catalog.gold.dim_customer (
        customer_sk INT,
        customer_key STRING,
        first_name STRING,
        last_name STRING,
        email STRING,
        phone STRING,
        address STRING,
        city STRING,
        state STRING,
        postal_code STRING,
        country STRING,
        company_name STRING,
        company_department STRING,
        job_title STRING,
        record_hash STRING,
        effective_start_date DATE,
        effective_end_date DATE,
        is_current BOOLEAN,
        dw_created_at TIMESTAMP,
        dw_updated_at TIMESTAMP
    )
    USING DELTA
""")
print("mia_catalog.gold.dim_customer ready")

mia_catalog.gold.dim_customer ready


### Cell 3 — helper functions (get/set watermark, get current version)

In [0]:
def get_last_processed_version(source_table: str, target_table: str) -> int:
    result = spark.sql(f"""
        SELECT last_processed_version
        FROM mia_catalog.gold._pipeline_state
        WHERE source_table = '{source_table}' AND target_table = '{target_table}'
    """).collect()
    if len(result) == 0:
        return -1
    return result[0]["last_processed_version"]

def set_last_processed_version(source_table: str, target_table: str, version: int):
    spark.sql(f"""
        MERGE INTO mia_catalog.gold._pipeline_state AS target
        USING (
            SELECT
                '{source_table}' AS source_table,
                '{target_table}' AS target_table,
                {version} AS last_processed_version,
                current_timestamp() AS updated_at
        ) AS source
        ON target.source_table = source.source_table
        AND target.target_table = source.target_table
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    
    
def get_current_table_version(table:str) -> int:
    return spark.sql(f"DESCRIBE HISTORY {table}").selectExpr("max(version)").collect()[0][0]   
    

### Cell 4 — backfill the watermark (run once)

In [0]:
dim_customer_row_count = spark.sql("SELECT count(*) as c FROM mia_catalog.gold.dim_customer").collect()[0]["c"]

if dim_customer_row_count == 0:
    print("dim_customer is empty — running initial full load")

    silver_customers_current = spark.table("mia_catalog.silver.silver_customers")
    window_spec = Window.orderBy("customer_key")

    dim_customer_initial = (
        silver_customers_current
        .select("customer_key", "first_name", "last_name", "email", "phone",
                 "address", "city", "state", "postal_code", "country",
                 "company_name", "company_department", "job_title", "record_hash")
        .withColumn("customer_sk", row_number().over(window_spec))
        .withColumn("effective_start_date", current_date())
        .withColumn("effective_end_date", lit("9999-12-31").cast("date"))
        .withColumn("is_current", lit(True))
        .withColumn("dw_created_at", current_timestamp())
        .withColumn("dw_updated_at", current_timestamp())
        .select("customer_sk", "customer_key", "first_name", "last_name", "email", "phone",
                 "address", "city", "state", "postal_code", "country",
                 "company_name", "company_department", "job_title", "record_hash",
                 "effective_start_date", "effective_end_date", "is_current",
                 "dw_created_at", "dw_updated_at")
    )

    dim_customer_initial.write.format("delta").mode("append").saveAsTable("mia_catalog.gold.dim_customer")
    print(f"Initial load complete: {dim_customer_initial.count()} rows")
else:
    print(f"dim_customer already has {dim_customer_row_count} rows — skipping initial load")

current_version_before_changes = get_current_table_version("mia_catalog.silver.silver_customers")
set_last_processed_version("mia_catalog.silver.silver_customers", "mia_catalog.gold.dim_customer", current_version_before_changes)
print(f"Watermark set at silver_customers version {current_version_before_changes}")

dim_customer already has 208 rows — skipping initial load
Watermark set at silver_customers version 1


### Cell 5 — pick 15 existing products to simulate a price change on

In [0]:
%sql
select * from mia_catalog.silver.silver_customers limit 5

customer_key,first_name,last_name,email,phone,username,birth_date,address,city,state,postal_code,country,company_name,company_department,job_title,record_hash,last_updated_ts
1,Emily,Johnson,emily.johnson@x.dummyjson.com,+81 965-431-3024,emilys,1996-5-30,626 Main Street,Phoenix,Mississippi,29112,United States,"Dooley, Kozey and Cronin",Engineering,Sales Manager,cd12fd7f0818b727fb9b636ce5154140,2026-08-12T08:18:22.058Z
2,Michael,Williams,michael.williams@x.dummyjson.com,+49 258-627-6644,michaelw,1989-8-10,385 Fifth Street,Houston,Alabama,38807,United States,Spinka - Dickinson,Support,Support Specialist,ca47f116c49fbb20432bd7006197833f,2026-08-12T08:18:22.058Z
3,Sophia,Brown,sophia.brown@x.dummyjson.com,+81 210-652-2785,sophiab,1982-11-6,1642 Ninth Street,Washington,Alabama,32822,United States,Schiller - Zieme,Research and Development,Accountant,f05d7a665327759ead00027061b22a45,2026-08-12T08:18:22.058Z
4,James,Davis,james.davis@x.dummyjson.com,+49 614-958-9364,jamesd,1979-5-4,238 Jefferson Street,Seattle,Pennsylvania,68354,United States,Pagac and Sons,Support,Research Analyst,15fb5333405ed1862a2ac0743d7113fe,2026-08-12T08:18:22.058Z
5,Emma,Miller,emma.miller@x.dummyjson.com,+91 759-776-1614,emmaj,1994-6-13,607 Fourth Street,Jacksonville,Colorado,26593,United States,Graham - Gulgowski,Human Resources,Quality Assurance Engineer,79ffb524d61f08ff95a550e5624574a1,2026-08-12T08:18:22.058Z


In [0]:
sample_customer_key = [i['customer_key'] 
                         for i in spark.sql("SELECT customer_key FROM mia_catalog.silver.silver_customers LIMIT 15").collect()]
print(f"Simulating price changes for {len(sample_customer_key)} existing products")
print(sample_customer_key)

Simulating price changes for 15 existing products
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15']


### Cell 6 — simulate a real change (optional but recommended, to exercise the incremental path)

In [0]:
sample_customer_keys = [row["customer_key"] for row in
    spark.sql("SELECT customer_key FROM mia_catalog.silver.silver_customers LIMIT 10").collect()]

print(f"Simulating a city change for {len(sample_customer_keys)} existing customers")
keys_sql_list = ", ".join([f"'{k}'" for k in sample_customer_keys])

spark.sql(f"""
    UPDATE mia_catalog.silver.silver_customers
    SET city = 'Relocated City',
        record_hash = md5(concat_ws('|', first_name, last_name, email, phone,
                                     address, 'Relocated City', state, postal_code,
                                     country, company_name, company_department, job_title)),
        last_updated_ts = current_timestamp()
    WHERE customer_key IN ({keys_sql_list})
""")
print("Simulated update applied to silver_customers")

Simulating a city change for 10 existing customers
Simulated update applied to silver_customers


### Cell 7 — confirm CDF captured the change

In [0]:
new_silver_version = get_current_table_version("mia_catalog.silver.silver_customers")
print(f"silver_customers is now at version {new_silver_version} (was {current_version_before_changes})")

display(spark.sql(f"""
    SELECT customer_key, city, _change_type, _commit_version
    FROM table_changes('mia_catalog.silver.silver_customers', {current_version_before_changes + 1}, {new_silver_version})
    ORDER BY _commit_version
"""))

silver_customers is now at version 2 (was 1)


customer_key,city,_change_type,_commit_version
2,Houston,update_preimage,2
6,Relocated City,update_postimage,2
2,Relocated City,update_postimage,2
8,Fort Worth,update_preimage,2
1,Relocated City,update_postimage,2
3,Washington,update_preimage,2
4,Relocated City,update_postimage,2
4,Seattle,update_preimage,2
5,Jacksonville,update_preimage,2
8,Relocated City,update_postimage,2


### Cell 8 — read the actual change rows to act on


In [0]:
silver_changes = spark.sql(f"""
    SELECT customer_key, first_name, last_name, email, phone, address, city,
           state, postal_code, country, company_name, company_department,
           job_title, record_hash
    FROM table_changes('mia_catalog.silver.silver_customers', {current_version_before_changes + 1}, {new_silver_version})
    WHERE _change_type IN ('insert', 'update_postimage')
""")
print(f"Changed/new rows to process into Gold: {silver_changes.count()}")

Changed/new rows to process into Gold: 10


### Cell 9 — split into changed vs. brand new customers

In [0]:
current_dim_customer = spark.table("mia_catalog.gold.dim_customer").filter(col("is_current") == True)

changed_customers = (
    silver_changes.alias("src")
    .join(
        current_dim_customer.select("customer_key", col("record_hash").alias("existing_hash")).alias("dim"),
        on="customer_key", how="inner"
    )
    .filter(col("record_hash") != col("existing_hash"))
    .select("src.*")
)

new_customers = (
    silver_changes.alias("src")
    .join(current_dim_customer.select("customer_key").alias("dim"), on="customer_key", how="left_anti")
)

print(f"Changed customers: {changed_customers.count()}")
print(f"Brand new customers: {new_customers.count()}")

Changed customers: 10
Brand new customers: 0


### Cell 10 — close out old versions

In [0]:
dim_customer_table = DeltaTable.forName(spark, "mia_catalog.gold.dim_customer")
changed_keys = [row["customer_key"] for row in changed_customers.select("customer_key").collect()]

if changed_keys:
    changed_keys_sql = ", ".join([f"'{k}'" for k in changed_keys])
    dim_customer_table.update(
        condition=f"customer_key IN ({changed_keys_sql}) AND is_current = true",
        set={
            "is_current": "false",
            "effective_end_date": "current_date()",
            "dw_updated_at": "current_timestamp()"
        }
    )
    print(f"Closed out {len(changed_keys)} old versions")
else:
    print("No changed customers this run")

Closed out 10 old versions


### Cell 11 — insert new versions

In [0]:
rows_to_insert = changed_customers.unionByName(new_customers)
insert_count = rows_to_insert.count()

if insert_count > 0:
    max_sk = spark.sql("SELECT COALESCE(MAX(customer_sk), 0) as max_sk FROM mia_catalog.gold.dim_customer").collect()[0]["max_sk"]
    window_spec = Window.orderBy("customer_key")

    rows_final = (
        rows_to_insert
        .withColumn("customer_sk", row_number().over(window_spec) + lit(max_sk))
        .withColumn("effective_start_date", current_date())
        .withColumn("effective_end_date", lit("9999-12-31").cast("date"))
        .withColumn("is_current", lit(True))
        .withColumn("dw_created_at", current_timestamp())
        .withColumn("dw_updated_at", current_timestamp())
        .select("customer_sk", "customer_key", "first_name", "last_name", "email", "phone",
                 "address", "city", "state", "postal_code", "country",
                 "company_name", "company_department", "job_title", "record_hash",
                 "effective_start_date", "effective_end_date", "is_current",
                 "dw_created_at", "dw_updated_at")
    )

    rows_final.write.format("delta").mode("append").saveAsTable("mia_catalog.gold.dim_customer")
    print(f"Inserted {insert_count} new version rows, surrogate keys starting from {max_sk + 1}")
else:
    print("Nothing to insert this run")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Inserted 10 new version rows, surrogate keys starting from 209


### Cell 12 — advance the watermark

In [0]:
set_last_processed_version("mia_catalog.silver.silver_customers", "mia_catalog.gold.dim_customer", new_silver_version)
print(f"Watermark advanced to silver_customers version {new_silver_version}")

Watermark advanced to silver_customers version 2


### Cell 13 — verify

In [0]:
display(spark.sql(f"""
    SELECT customer_sk, customer_key, city, is_current, effective_start_date, effective_end_date
    FROM mia_catalog.gold.dim_customer
    WHERE customer_key = '{sample_customer_keys[0]}'
    ORDER BY customer_sk
"""))

display(spark.sql("""
    SELECT is_current, count(*) as row_count
    FROM mia_catalog.gold.dim_customer
    GROUP BY is_current
"""))

customer_sk,customer_key,city,is_current,effective_start_date,effective_end_date
1,1,Phoenix,false,2026-08-15,2026-08-16
209,1,Relocated City,true,2026-08-16,9999-12-31


is_current,row_count
true,208
false,10
